# Lab: Advanced Propositional Logic Inference Engine

## Case Study: Smart Home Security Reasoning System

### Scenario

You are building a **smart home security system** that uses logical reasoning to decide when to trigger alerts, lock doors, and notify owners. The system must reason about multiple sensors and rules to ensure safety.

In this lab, you will implement:

- Logical inference rules (Modus Ponens, etc.)
- Model checking
- CNF conversion
- Resolution-based theorem proving

---

## New Capabilities

### 1. Inference Rules
- Modus Ponens
- And-Elimination
- Implication Elimination
- Biconditional Elimination
- Double Negation
- De Morgan’s Laws
- Distributive Laws

### 2. Model Checking
- Evaluate if a query is true in **all models**

### 3. CNF Conversion
- Convert any formula into **Conjunctive Normal Form (CNF)**

### 4. Resolution
- Apply resolution rule
- Use proof by contradiction to prove queries

---

## Input Files

### knowledge_base.txt

Each line contains a fact or a logical rule.


### query.txt

Single query:





---

## Learning Objectives

By completing this lab, you will:

- Apply inference rules programmatically
- Convert formulas to CNF
- Implement resolution-based theorem proving
- Perform model checking
- Solve complex logic queries automatically


## Step 1: Knowledge Base Representation

Create a structure to store logical sentences.

You will:
- Store all facts and rules
- Implement a method to check if a query is entailed

In [1]:
class KnowledgeBase:
    def __init__(self):
        self.rules = []

    def add(self, rule):
        self.rules.append(rule)

    def entails(self, query):
        """
        Placeholder for checking if KB entails query
        Can be implemented with:
        - Model checking
        - Resolution
        """
        return model_check(self, query)
        

## Step 2: Implement Inference Rules
You will implement core logical inference rules.

These rules help derive new knowledge from existing facts.

In [2]:
class InferenceEngine:

    def apply_modus_ponens(self, kb):
        """
        If P and (P => Q) exist, infer Q
        """

        inferred = set()
        facts = set()
        implications = []

        for rule in kb.rules:
            if "=>" in rule:
                implications.append(rule)
            else:
                facts.add(rule)

        for rule in implications:
            left, right = rule.split("=>")
            left = left.strip()
            right = right.strip()

            conditions = [cond.strip() for cond in left.split("AND")]

            if all(c in facts for c in conditions):
                inferred.add(right)


    def eliminate_and(self, sentence):
        """
        From (A AND B), infer A and B
        """
        if "AND" in sentence:
            parts = sentence.split("AND")
            return [p.strip() for p in parts]
        return [sentence]

    def remove_implication(self, sentence):
        """
        Replace (P => Q) with (NOT P OR Q)
        """

        if "=>" in sentence:
            left, right = sentence.split("=>")
            return f"NOT {left.strip()} OR {right.split()}"
        return sentence

    def simplify_biconditional(self, sentence):
        """
        Replace (P <=> Q) with two implications
        """

        if "<=>" in sentence:
            left, right = sentence.split("<=>")
            left = left.strip()
            right = right.strip()
            return f"{left} => {right}) AND ({right} => {left}"
        return sentence

    def remove_double_negation(self, sentence):
        """
        Simplify NOT NOT P into P
        """
        return sentence.replace("NOT NOT", "").strip()

    def apply_de_morgan(self, sentence):
        """
        Push NOT inward using De Morgan laws
        """
        sentence = sentence.strip()

        if sentence.startswith("NOT (") and sentence.endswith(")"):
            inner = sentence[5:-1]

            if "AND" in inner:
                parts = inner.split("AND")
                return " OR ".join([f"NOT {p.strip()}" for p in parts])

            elif "OR" in inner:
                parts = inner.split("OR")
                return " AND ".join([f"NOT {p.strip()}" for p in parts])

        return sentence

    def apply_distribution(self, sentence):
        """
        Apply distributive laws for CNF conversion
        """
        if "OR" in sentence and "AND" in sentence:
            parts = sentence.split("OR")
            A = parts[0].strip()
            rest = parts[1].strip()

            if rest.startswith("(") and rest.endswith(")"):
                inner = rest[1:-1]

                if "AND" in inner:
                    B, C = inner.split("AND")
                    B = B.strip()
                    C = C.strip()

                    return f"({A} OR {B}) AND ({A} OR {C})"

        return sentence

## Step 3: Convert to CNF

Resolution requires all formulas to be in **Conjunctive Normal Form (CNF)**.

You will implement a pipeline to transform any formula into CNF.

In [3]:
class CNFTransformer:

    def convert(self, expression):
        """
        Steps:
        1. Remove biconditionals
        2. Remove implications
        3. Push NOT inward
        4. Apply distributive laws
        5. Flatten into conjunction of disjunctions
        """

        engine = InferenceEngine()

        expression = engine.simplify_biconditional(expression)
        expression = engine.remove_implication(expression)
        expression = engine.apply_de_morgan(expression)
        expression = engine.apply_distribution(expression)

        return expression

## Step 4: Resolution-Based Theorem Proving

Use the resolution rule to determine if a query is entailed.

This method uses **proof by contradiction**.

In [4]:
class ResolutionEngine:

    def resolve_clauses(self, c1, c2):
        """
        Combine clauses by eliminating complementary literals
        """
        resolvents = []

        for lit in c1:
            if f"NOT {lit}" in c2:
                new_clause = (c1 - {lit}) | (c2 - {f"NOT {lit}"})
                resolvents.append(new_clause)

        return resolvents


    def infer(self, kb, query):
        """
        - Convert KB to CNF
        - Add NOT query
        - Repeatedly resolve clauses
        - Return True if contradiction found
        """
        cnf_engine = CNFTransformer()
        clauses = []

        rules = kb.rules
        cnf_rules = [cnf_engine.convert(r) for r in rules]

        for rule in cnf_rules:
            for clause_str in rule.split("AND"):
                clause_str = clause_str.strip().strip("()")
                facts = set(lit.strip() for lit in clause_str.split("OR"))
                clauses.append(facts)

        clauses.append({f"NOT {query}"})

        while True:
            n = len(clauses)
            pairs = [(clauses[i], clauses[j]) for i in range(n) for j in range(i+1, n)]

            resolvents_found = []

            for (c1, c2) in pairs:
                resolvents = self.resolve_clauses(c1, c2)
                for res in resolvents:
                    if not res:
                        return True
                    resolvents_found.append(res)

            resolvents_found = [r for r in resolvents_found if r not in clauses]
            if not resolvents_found:
                return False

            clauses.extend(resolvents_found)

## Step 5: Model Checking

Model checking verifies if a query is true in **all possible models**.

In [5]:
def model_check(kb, query):
    """
    - Extract all symbols
    - Generate all possible truth assignments
    - Verify KB entails query in all models
    """
    
    cnf_engine = CNFTransformer()

    rules = kb.rules
    cnf_rules = [cnf_engine.convert(r) for r in rules]

    symbols = set()
    for rule in cnf_rules:
        words = rule.replace("(", "").replace(")", "").split()
        for w in words:
            if w not in ["AND", "OR", "NOT"]:
                symbols.add(w)
    symbols = list(symbols)

    from itertools import product
    for values in product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, values))

        kb_true = True
        for rule in cnf_rules:
            expr_eval = rule.replace("AND", "and").replace("OR", "or").replace("NOT", "not")
            for var in model:
                expr_eval = expr_eval.replace(var, str(model[var]))
            if not eval(expr_eval):
                kb_true = False
                break
        if kb_true:
            if not model.get(query, False):
                return False

    return True

## Step 6: File Input

Read the knowledge base and query from files.

In [6]:
def load_kb(file):
    kb = KnowledgeBase()

    with open(file, 'r') as f:
        for line in f:
            if not line.strip() or line.startswith("#"):
                continue
        
            # if any(word in line for word in ["AND", "NOT", "OR", "=>", "<=>"]):
            #     kb.add(line)
            kb.add(line)

    return kb

def load_query(file):
    with open(file, 'r') as f:
        query = f.read().strip()
        return query

## Step 7: Output Writer
Do not modify this function.

In [7]:
def write_output(results, filename="output.txt"):
    output = "Inference Results:\n"
    output += "-------------------\n"

    for method, result in results.items():
        output += f"{method}: {result}\n"

    print(output)

    with open(filename, "w") as f:
        f.write(output)

## Step 8: Main Execution

Run both inference methods:

- Model Checking
- Resolution

Compare their results.

In [8]:
def main():
    kb = load_kb("knowledge_base.txt")
    query = load_query("query.txt")

    results = {}

    # Run Model Checking
    results["Model Checking"] = model_check(kb, query)

    # Run Resolution
    engine = ResolutionEngine()
    results["Resolution"] = engine.infer(kb, query)

    write_output(results)

if __name__ == "__main__":
    main()

Inference Results:
-------------------
Model Checking: False
Resolution: True



## Step 9: Logic Puzzle

### Problem

1. If motion is detected, the alarm turns on  
2. If the alarm turns on, the owner is notified  
3. If a door is opened at night, the alarm turns on  
4. Motion is detected  
5. It is night  
6. A window is open  

**Query:** Is the owner notified?

### Tasks

1. Convert the scenario into logical expressions
2. Add to knowledge base
3. Solve using:
   - Model checking
   - Resolution

knowledge base:

MotionDetected => AlarmOn
DoorOpen AND Night => AlarmOn
AlarmOn => NotifyOwner

MotionDetected
Night
WindowOpen

Resolution:

MotionDetected so AlarmOn
AlarmOn so NotifyOwner

hence Owner notified, rule 3 don't effect our reasoning

## Step 10: Conceptual Questions

1. Why must logical expressions be converted to CNF before resolution?

2. Compare model checking and resolution in terms of efficiency.

3. What is proof by contradiction?

4. Why does model checking scale poorly with many symbols?

5. How do inference rules improve reasoning efficiency?

In [9]:
# Answers
# They need to be converted to CNF before resolution as resolution uses clauses and resolves them complementary basis so without CNF they expressions can be complex and ambiguous making resolution not possioble
# Model checking is slow and grows exponentially as KB and symbols increase while resolution often works more efficiently especially when having larger KB and only needing a proof
# It means that we assume the statement is false and then try to shww that it leads to a contradiction 
# Because it requires assigning all combinations of true, false to the symbols resulting in 2^n combinations for n symbols
# THey help simplify the expressions and derive new facts without finding all models and avoid redundant computation